# Data Profiling — Sales Project

## Purpose

Understand the condition of the Bronze data before defining validation rules and making cleaning decisions. This notebook produces summary reports and examples of potential issues without modifying the saved tables.

## Tables Covered

- **Sales:** order records, quantities, amounts and dates.
- **Customers:** customer IDs, names, gender and cities.
- **Products:** product IDs, names, prices and categories.
- **Stores:** store IDs, cities, states and locations.

## Profiling Approach

For each table, apply the relevant checks:

- **Overview:** inspect row counts, columns, data types and sample records.
- **Completeness:** count nulls and blank values.
- **Uniqueness:** examine exact duplicates and repeated IDs; investigate conflicting records where applicable.
- **Numeric profiling:** temporarily interpret numbers and review conversion failures, ranges, averages and nonpositive values.
- **Date profiling:** examine missing, unreadable and future dates.
- **Categorical profiling:** review distinct values and their frequencies.
- **Text formatting:** identify leading and trailing spaces.

Business checks exclude ingestion metadata. Temporary conversions preserve the original business values.

## Expected Outputs

- Summary reports showing missing values, duplicates and value distributions.
- Examples of records requiring further investigation.
- Findings that guide the next stage: data validation.

## Scope

Reference-table profiling is excluded from this notebook. The reference tables will be used during validation to check approved categories, IDs and location relationships.

No records are corrected, removed or written back to Bronze.

## Sales Profiling

Examine the Bronze Sales table for missing values, duplicates, numeric issues and date issues. Keep the saved data unchanged.

### Load and Inspect the Data

- Import the PySpark functions used throughout the notebook.
- Set the catalog and schema, then read the saved Sales table.
- Count all records and stop if the table is empty.
- Use the total row count as the basis for later profiling percentages.
- Inspect stored data types and preview ten records in their original source order.

In [0]:
#import pyspark built in sql functions 
from pyspark.sql import functions as F

# Set the location of our existing Bronze table.
CATALOG = "sales_quality"
SCHEMA = "bronze_layer"

# Read the saved table into a DataFrame.
#Here we read the saved tabel not the excel file 
sales = spark.table(f"{CATALOG}.{SCHEMA}.sales")

#count the records before calcualting the profiling metrics . we treate the total row couts as reference point for other calcuations 
total_rows = sales.count()
if total_rows == 0 :
    raise ValueError("The sales table is impty review the ingestion")

print(f"Total rows: {total_rows:,}")

# we will exclude the _source_row_number column . and call the other columns as business columns 
business_columns = [
    column for column in sales.columns 
    if not column.startswith("_")
]

#inspect data types currently stored in Bronze 
sales.printSchema()

# Preview business values in their original Excel row order.
display(
    sales
    .orderBy("_source_row_number")
    .select(*business_columns)
    .limit(10)
)

Total rows: 2,004
root
 |-- date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- units_sold: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- _source_row_number: long (nullable = true)



date,order_id,customer_id,product_id,units_sold,total_amount,store_id
2021-08-23 00:00:00,ORD000001,CS_Sim_5,P10007,5,315,St_Pho_ A
2022-01-16 00:00:00,ORD000002,CS_Dav_37,P10073,4,136,St_Las_Ne
2021-07-19 00:00:00,ORD000003,CS_Dav_37,P10037,4,128,St_Las_Ne
2022-03-30 00:00:00,ORD000004,CS_Mat_12,P10082,2,60,St_Las_Ne
2022-01-29 00:00:00,ORD000005,CS_Bea_11,P10085,5,170,St_Atl_Ge
2020-12-22 00:00:00,ORD000006,CS_Kel_44,P10030,5,160,St_Hou_Te
2020-08-30 00:00:00,ORD000007,CS_Dav_15,P10037,6,192,St_Det_Mi
2021-01-24 00:00:00,ORD000008,CS_Don_33,P10031,5,170,St_Cha_No
2021-07-24 00:00:00,ORD000009,CS_Cal_22,P10066,3,96,St_Pho_ A
2022-06-04 00:00:00,ORD000010,CS_Com_17,P10032,6,198,St_Ind_In


### Missing Values

- Check each business column for actual nulls and blank text.
- Count nulls and blanks separately.
- Calculate the total missing values and their percentage of all records.
- Display one summary row per column, showing the highest missing counts first.
- Use these findings to prepare completeness rules during validation.

The Bronze data remains unchanged.

In [0]:


# Collect one summary row for each business column.
missing_results = []
for column in business_columns:
    # Identify actual null values.it goes through each row on the specified columns and then retur true if null 
    is_null = F.col(column).isNull()
    #Identify empty text or text containing only spaces.
    is_blank = (
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) == "")
    )
    #count each one seperately 
    null_count = sales.filter(is_null).count()
    blank_count = sales.filter(is_blank).count()
    # Calculate the total and percentage of missing values.
    missing_count = null_count + blank_count
    missing_pct = round(missing_count / total_rows * 100, 2)

    # Store the results for this column.
    missing_results.append(
        (column, null_count, blank_count, missing_count, missing_pct)
    )

# Build the report once, after the loop finishes.
missing_profile = spark.createDataFrame(
    missing_results,
    """
    column_name string,
    null_count long,
    blank_count long,
    missing_count long,
    missing_pct double
    """,
)

# Show columns with the most missing values first.
display(
    missing_profile.orderBy(
        F.desc("missing_count"),
        "column_name",
    )
)

column_name,null_count,blank_count,missing_count,missing_pct
customer_id,4,0,4,0.2
units_sold,4,0,4,0.2
date,3,0,3,0.15
order_id,2,0,2,0.1
product_id,0,0,0,0.0
store_id,0,0,0,0.0
total_amount,0,0,0,0.0


### Exact Duplicate Rows

- Compare records using business columns only, excluding ingestion metadata.
- Count distinct combinations of all business values.
- Subtract the distinct count from the total to find extra copies beyond the first occurrence.
- Display total rows, distinct business rows and extra duplicate rows.

This identifies completely identical records. Repeated order IDs will be examined separately. No records are removed from Bronze.

In [0]:
# Select only the seven business columns.
business_data = sales.select(*business_columns)

#count distinct combination of all business values 
unique_rows = business_data.distinct().count()

#count copies or duplicated rows 
exact_duplicate_rows = total_rows - unique_rows 
print(f"Total rows: {total_rows:,}")
print(f"Distinct business rows: {unique_rows:,}")
print(f"Extra duplicate rows: {exact_duplicate_rows:,}")

Total rows: 2,004
Distinct business rows: 1,999
Extra duplicate rows: 5


### Repeated Order IDs

- Exclude records with null or blank order IDs from this check.
- Count occurrences of each populated order ID.
- Display IDs appearing more than once.
- Retrieve their business records to compare the values.
- Use these examples to investigate identical copies or conflicting details before cleaning.

A repeated order ID does not automatically mean its records contain different business values. The Bronze data remains unchanged.

In [0]:
# now we need to find different business versions of same order id , excluding null order id rows 
identified_sales = sales.filter(
    F.col("order_id").isNotNull()
    &(F.trim(F.col("order_id")) != "")
)
#count how many times each order id appears 
repeated_orders = (
    identified_sales
    .groupBy("order_id")
    .count()
    .filter(F.col("count")>1)
)

#show repeated ids and their number of occurrances 
display(
    repeated_orders.orderBy(
        F.desc("count"),
        "order_id",
    )
)

# we need to inspect the records behind those ids . to compare business values before deciding how to handle them during the cleaning 
duplicate_examples = (
    identified_sales
    .join(
        repeated_orders.select("order_id"),
        on= "order_id",
        how = "inner"
    )
    .select(*business_columns)
)

#display those repeated ordes with their business columns only 
display(
    duplicate_examples.orderBy("order_id")
)

order_id,count
ORD000691,2
ORD000805,2
ORD001014,2
ORD001587,2
ORD001802,2


date,order_id,customer_id,product_id,units_sold,total_amount,store_id
2021-04-25 00:00:00,ORD000691,CS_Jar_24,P10053,5,175,St_Los_Ca
2021-04-25 00:00:00,ORD000691,CS_Jar_24,P10053,5,175,St_Los_Ca
2020-02-19 00:00:00,ORD000805,CS_Liv_35,P10073,3,102,St_Alb_ N
2020-02-19 00:00:00,ORD000805,CS_Liv_35,P10073,3,102,St_Alb_ N
2020-04-28 00:00:00,ORD001014,CS_Bel_43,P10061,6,192,St_Pho_ A
2020-04-28 00:00:00,ORD001014,CS_Bel_43,P10061,6,192,St_Pho_ A
2021-06-13 00:00:00,ORD001587,CS_Stu_47,P10013,6,474,St_Cha_No
2021-06-13 00:00:00,ORD001587,CS_Stu_47,P10013,6,474,St_Cha_No
2020-10-10 00:00:00,ORD001802,CS_Sim_26,P10098,3,45,St_Col_Oh
2020-10-10 00:00:00,ORD001802,CS_Sim_26,P10098,3,45,St_Col_Oh


### Numeric Profiling

- Examine `units_sold` and `total_amount` using business columns only.
- Create temporary numeric versions with `try_cast`; failed conversions become null.
- Display original and converted values for comparison.
- Calculate non-null numeric counts, minimums, maximums and averages.
- Keep negative, zero and extreme values included for later investigation.

The original values and saved Bronze table remain unchanged. Successful conversion does not guarantee that a value meets business rules.

#### Prepare Numeric Values

Create temporary numeric columns and compare them with the original text. These columns will support numeric summaries and conversion-failure checks.

In [0]:
# Start with business columns only.
numeric_data = sales.select(*business_columns)
# we will add another column units_sold_numeric and try to cast it to Double . using try_cast function  which ruturn null if the cast fails 
#F.expr : this expect Spark SQL expression 
numeric_data = (
    numeric_data
    .withColumn(
        "units_sold_numeric",
        F.expr("try_cast(units_sold AS DOUBLE)"),
    )
    .withColumn(
        "total_amount_numeric",
        F.expr("try_cast(total_amount AS DOUBLE)"),
    )
    )
#shows only columns we need to compare with order id as well 
display(
    numeric_data.select(
        "order_id",
        "units_sold",
        "units_sold_numeric",
        "total_amount",
        "total_amount_numeric"
    )
)



order_id,units_sold,units_sold_numeric,total_amount,total_amount_numeric
ORD001003,5,5.0,170,170.0
ORD001004,4,4.0,132,132.0
ORD001005,6,6.0,102,102.0
ORD001006,5,5.0,165,165.0
ORD001007,5,5.0,175,175.0
ORD001008,3,3.0,105,105.0
ORD001009,2,2.0,64,64.0
ORD001010,2,2.0,68,68.0
ORD001011,2,2.0,66,66.0
ORD001012,3,3.0,96,96.0


#### Summarize Numeric Values

Calculate the count, minimum, maximum and average of converted values. Nulls are excluded, while negative, zero and extreme values remain included.

In [0]:
# Summarize the temporary numeric columns.This shows the number of successfully converted values, minimum, maximum and average. Negative values, zeros and large quantities remain included for now. These findings will help us define validation rules and investigate outliers later.
numeric_summary=(
    numeric_data
    .select(
        F.col("units_sold_numeric").alias("units_sold"),
        F.col("total_amount_numeric").alias("total_amount"),
    )
    .summary("count", #count here exclude null. but it include values failed to conver 
             "min","max","mean")
    )

print("\nTHOSE ARE SUMMARY RESULSTS : ")
display(numeric_summary)


THOSE ARE SUMMARY RESULSTS : 


summary,units_sold,total_amount
count,1998,2004
min,-3.0,20.0
max,800.0,50700.0
mean,4.965965965965966,184.82035928143713


#### Inspect Conversion Failures and Nonpositive Values

- Examine each numeric field separately.
- Identify populated values that failed numeric conversion.
- Identify converted values that are zero or negative.
- Display the order ID, original value and converted value for investigation.
- Exclude nulls and blanks from conversion failures because completeness checks cover them.

Use these examples to prepare numeric validation rules. No values are corrected or removed.

In [0]:
# Inspect each numeric field separately.
for column in ["units_sold", "total_amount"]:
    parsed_column = f"{column}_numeric"
    # A populated value that could not be converted to a number.
    conversion_failed = (
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) != "")
        & F.col(parsed_column).isNull()
    )
    # A value that converted successfully but is zero or negative.
    nonpositive = F.col(parsed_column) <= 0
    print(f"Values to review in {column}:")
    display(
        numeric_data
        .filter(conversion_failed | nonpositive)
        .select(
            "order_id",
            column,
            parsed_column,
        )
        .orderBy("order_id")
    )

Values to review in units_sold:


order_id,units_sold,units_sold_numeric
ORD000291,0,0.0
ORD000304,three,null
ORD000592,-3,-3.0
ORD000695,-2,-2.0
ORD000845,0,0.0
ORD001349,five,null


Values to review in total_amount:


order_id,total_amount,total_amount_numeric


### Date Profiling

Examine the Sales date column for missing, unreadable and future values. Use temporary date conversion to produce summaries and examples without changing the original values or the Bronze table.

#### Prepare Date Values

- Select business columns only.
- Create a temporary `parsed_date` column using `try_cast`.
- Keep the original text alongside the converted date.
- Preview both values for comparison.

Failed conversions become null. The temporary column will support the following date checks.

In [0]:
# Select business columns only.
date_data = sales.select(*business_columns)

# Temporarily interpret the original text as a date.
date_data =date_data.withColumn (
    "parsed_date",
    F.expr("try_cast(`date` AS DATE)")
)
#Compare original values . with interpreted dates. 
display(
    date_data.select(
        "order_id",
        "date",
        "parsed_date"
    ).limit(10)
)


order_id,date,parsed_date
ORD001003,2021-04-20 00:00:00,2021-04-20
ORD001004,2021-06-30 00:00:00,2021-06-30
ORD001005,2020-05-18 00:00:00,2020-05-18
ORD001006,2020-09-09 00:00:00,2020-09-09
ORD001007,2020-02-13 00:00:00,2020-02-13
ORD001008,2021-01-30 00:00:00,2021-01-30
ORD001009,2020-10-07 00:00:00,2020-10-07
ORD001010,2022-04-12 00:00:00,2022-04-12
ORD001011,2022-02-26 00:00:00,2022-02-26
ORD001012,2021-05-15 00:00:00,2021-05-15


#### Summarize Date Values

- Count missing dates, populated values that failed conversion and readable dates.
- Count dates later than today in the Spark session’s timezone.
- Display the earliest and latest readable dates.

Readable dates include future dates. Missing values and failed conversions are excluded from the earliest and latest dates.

In [0]:
#now we need to summerize the results 
#identify nulls and blak text.
missing_date = (
    F.col("date").isNull()
    | (F.trim(F.col("date"))=="")
)

# Identify populated values that could not become dates.
unreadable_date = (
    ~missing_date
    & F.col("parsed_date").isNull()
)

# Compare dates with today's date in the Spark session's timezone.
future_date = F.col("parsed_date") > F.current_date()


# Produce one summary row for the date column.
date_summary = date_data.agg(
    F.count(F.when(missing_date, 1)).alias("missing_dates"),
    F.count(F.when(unreadable_date, 1)).alias("unreadable_dates"),
    F.count("parsed_date").alias("readable_dates"),
    F.count(F.when(future_date, 1)).alias("future_dates"),
    F.min("parsed_date").alias("earliest_date"),
    F.max("parsed_date").alias("latest_date"),
)

display(date_summary)




missing_dates,unreadable_dates,readable_dates,future_dates,earliest_date,latest_date
3,3,1998,3,2020-01-01,2099-01-03


#### Inspect Date Issues

- Label affected records as missing, unreadable or future dates.
- Display the order ID, original value, converted date and issue.
- Use these examples to prepare date-validation rules.

No records are corrected or removed.

In [0]:
# Assign a descriptive label to each date issue.
date_examples = (
    date_data
    .withColumn(
        "date_issue",
        F.when(missing_date, "Missing date")
        .when(unreadable_date, "Unreadable date")
        .when(future_date, "Future date"),
    )
    # Keep only records with a date finding.
    .filter(F.col("date_issue").isNotNull())
    .select(
        "order_id",
        "date",
        "parsed_date",
        "date_issue",
    )
)

display(
    date_examples.orderBy("date_issue", "order_id")
)

order_id,date,parsed_date,date_issue
ORD000113,2099-01-02 00:00:00,2099-01-02,Future date
ORD000188,2099-01-01 00:00:00,2099-01-01,Future date
ORD000992,2099-01-03 00:00:00,2099-01-03,Future date
ORD000904,null,null,Missing date
ORD001106,null,null,Missing date
ORD001519,null,null,Missing date
ORD000664,2020-04-31,null,Unreadable date
ORD000853,2022-02-30,null,Unreadable date
ORD001895,2021-13-10,null,Unreadable date


## Customers Profiling

Examine customer records for missing values, duplicates, category distributions and text-formatting issues. Use the findings to prepare validation rules while keeping Bronze unchanged.

### Load and Inspect the Data

- Read the Customers table from Bronze.
- Select business columns only, excluding ingestion metadata.
- Count records and stop if the table is empty.
- Use the row count as the basis for later profiling percentages.
- Inspect stored data types and preview ten records ordered by customer ID.

In [0]:
# Read the existing Customers table.
customers = spark.table(f"{CATALOG}.{SCHEMA}.customers")

# Keep business columns only, excluding ingestion metadata.
customer_columns = [
    column for column in customers.columns
    if not column.startswith("_")
]
customers = customers.select(*customer_columns)
# Count records before calculating profiling metrics.
customer_total_rows = customers.count()
if customer_total_rows == 0:
    raise ValueError("The Customers table is empty. Review ingestion first.")

print(f"Total customers: {customer_total_rows}")
print(f"Business columns: {len(customer_columns)}")

# Inspect the data types stored in Bronze.
customers.printSchema()

# Preview customer records in a consistent order.
display(
    customers.orderBy("customer_id").limit(10)
)


Total customers: 49
Business columns: 5
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)



customer_id,first_name,last_name,gender,city
CS_All_50,Allison,Connie,F,Albuquerque
CS_Ash_36,Ashley,Emma,F,Houston
CS_Ate_41,Atencio,Morris,M,Detroit
CS_Bar_13,Barajas,John,M,Columbus
CS_Bea_11,Beard,Robert,M,Indianapolis
CS_Bel_16,Bell,Thomas,M,Charlotte
CS_Bel_43,Bell,Angelo,M,Oklahoma City
CS_Bry_48,Bryant,John,M,Oklahoma City
CS_Buf_8,Buford,Ralph,M,Oklahoma City
CS_Bur_28,Burton,Theodore,M,Miami


### Missing and Distinct Values

- Count nulls and blank text in each business column.
- Calculate total missing values and their percentage of customer records.
- Count distinct populated values exactly as stored.
- Display columns with the highest missing counts first.

Use these findings to prepare completeness rules and understand column uniqueness.

In [0]:
#We’ll check completeness, duplicates, category distributions and text formatting. These checks describe the data; they do not modify Bronze.

#check columns of customers for missing values 
customer_missing_results = []
for column in customer_columns:
    is_null = F.col(column).isNull()
    is_blank = (
        F.col(column).isNotNull()
        &(F.trim(F.col(column)) == "")
    )
    null_count = customers.filter(is_null).count()
    blank_count= customers.filter(is_blank).count()
    missing_count = null_count + blank_count 
    #counts the number of different non-empty values in one column.
    distinct_count = (
        customers
        .filter(~(is_null | is_blank))
        .select(column)
        .distinct()
        .count()
    )

    customer_missing_results.append((
        column,
        null_count,
        blank_count,
        missing_count,
        round(missing_count / customer_total_rows *100,2),
        distinct_count
    ))

# Create one report row per business column.
customer_column_profile = spark.createDataFrame(
    customer_missing_results,
    """
    column_name string,
    null_count long,
    blank_count long,
    missing_count long,
    missing_pct double,
    distinct_values long
    """,
)


print("This show missing Profilling")
display(
    customer_column_profile.orderBy(
        F.desc("missing_count"),
        "column_name",
    )
)

This show missing Profilling


column_name,null_count,blank_count,missing_count,missing_pct,distinct_values
city,0,0,0,0.0,14
customer_id,0,0,0,0.0,49
first_name,0,0,0,0.0,46
gender,0,0,0,0.0,2
last_name,0,0,0,0.0,43


### Duplicate Records and Customer IDs

- Count extra copies of completely identical business records.
- Exclude missing customer IDs from repeated-ID checks.
- Identify customer IDs appearing more than once.
- Identify IDs associated with different business records.
- Display the three duplicate measures.

These findings help distinguish identical copies from conflicting customer details. No records are removed.

In [0]:
#check exact duplicates and repeated customer ID

#count copies of identical business records 
customer_unique_rows = customers.distinct().count() # should equal 49 as we have 49 unique customers
customer_exact_duplicates = (
    customer_total_rows - customer_unique_rows
) 
# print(customer_exact_duplicates)  # now we are done with duplicates . ther is no duplicates 

# we need to check repeated customer id 
identified_customers = customers.filter(
    F.col("customer_id").isNotNull()
    &(F.trim(F.col("customer_id")) != "")

)
 
#find customer IDs appearing more than once 
repeated_customer_ids = (
    identified_customers
    .groupBy("customer_id")
    .count()
    .filter(F.col("count")>1)
)

#this code check if there is different business version for the same customer 
# Remove identical copies temporarily to find different
# business records sharing the same customer ID.
conflicting_customer_ids = (
    identified_customers
    .distinct()
    .groupBy("customer_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
    .filter(F.col("distinct_versions") > 1)
) # actually there is no confilicting rows as we know that 

# Summarize the three duplicate measures.
customer_duplicate_profile = spark.createDataFrame(
    [
        ("extra_exact_duplicate_rows", customer_exact_duplicates),
        ("repeated_customer_id_groups", repeated_customer_ids.count()),
        ("conflicting_customer_id_groups", conflicting_customer_ids.count()),
    ],
    "metric string, value long",
)

print("This show Numeric Profilling for uplicates and repeated customer ID")

display(customer_duplicate_profile)

This show Numeric Profilling for uplicates and repeated customer ID


metric,value
extra_exact_duplicate_rows,0
repeated_customer_id_groups,0
conflicting_customer_id_groups,0


### Inspect Repeated Customer Records

- Retrieve the business records belonging to repeated customer IDs.
- Display them together to compare names and other details.
- Use the examples to investigate duplicates before deciding how to handle them.

An empty result means no populated customer IDs were repeated.

In [0]:
#we can inspect those records with repeated id if there using this code 
# Retrieve the business records behind repeated IDs.
customer_duplicate_examples = (
    identified_customers
    .join(
        repeated_customer_ids.select("customer_id"),
        on="customer_id",
        how="inner",
    )
    .select(*customer_columns)
)

display(
    customer_duplicate_examples.orderBy(
        "customer_id",
        "first_name",
        "last_name",
    )
)


customer_id,first_name,last_name,gender,city


### Gender and City Distributions

- Group records by each stored gender and city value.
- Calculate counts and percentages of all customer records.
- Display the most frequent values first.
- Review the results for unexpected labels or inconsistent spelling.

Category frequencies describe the data; they do not establish whether a value is valid.

In [0]:
#check gender and city distributions
# Examine each categorical column separately.
for column in ["gender", "city"]:

    customer_category_profile = (
        customers
        .groupBy(column)
        .count()
        .withColumnRenamed("count", "row_count")
        .withColumn(
            "row_pct",
            F.round(
                F.col("row_count") / F.lit(customer_total_rows) * 100,
                2,
            ),
        )
    )

    print(f"Distribution of {column}:")

    display(
        customer_category_profile.orderBy(
            F.desc("row_count"),
            column,
        )
    )


Distribution of gender:


gender,row_count,row_pct
M,29,59.18
F,20,40.82


Distribution of city:


city,row_count,row_pct
Detroit,8,16.33
Columbus,6,12.24
Oklahoma City,6,12.24
Baltimore,5,10.2
Charlotte,4,8.16
Houston,3,6.12
Las Vegas,3,6.12
Seattle,3,6.12
Albuquerque,2,4.08
Atlanta,2,4.08


### Leading and Trailing Spaces

- Check populated business values for spaces at the beginning or end.
- Exclude nulls and blank-only values already covered by completeness checks.
- Count affected records and display examples when found.
- Show a trimmed preview while preserving the original values.

Use these findings to prepare text-standardization rules. Internal spaces and the Bronze data remain unchanged.

In [0]:
#check leading and trailing spaces 
customer_space_results = []
for column in customer_columns:

    # Compare each populated value with its trimmed version.
    # Blank-only values are already covered by missing profiling.
    has_outer_spaces = (
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) != "")
        & (F.col(column) != F.trim(F.col(column)))
    )

    affected_rows = customers.filter(has_outer_spaces).count()

    customer_space_results.append(
        (column, affected_rows)
    )

    # Show examples only when a formatting issue exists.
    if affected_rows > 0:
        print(f"Outer spaces found in {column}:")

        display(
            customers
            .filter(has_outer_spaces)
            .select(
                F.col("customer_id").alias("record_id"),
                F.col(column).alias("original_value"),
                F.trim(F.col(column)).alias("trimmed_preview"),
            )
            .limit(10)
        )

# Display the formatting summary.
customer_space_profile = spark.createDataFrame(
    customer_space_results,
    "column_name string, rows_with_outer_spaces long",
)

display(
    customer_space_profile.orderBy(
        F.desc("rows_with_outer_spaces"),
        "column_name",
    )
) # we would expect zero issues .on this check 


column_name,rows_with_outer_spaces
city,0
customer_id,0
first_name,0
gender,0
last_name,0


## Products Profiling

Examine product records for missing values, duplicates, price issues, category inconsistencies and text-formatting problems.

Use the findings to prepare product validation rules. Business columns are checked without modifying the Bronze table.

### Load and Inspect the Data

- Read the Products table from Bronze.
- Select business columns only.
- Count records and stop if the table is empty.
- Inspect stored data types and preview ten products.
- Reuse the DataFrame and row count in the following checks.

In [0]:
# Read the existing Bronze table.
products = spark.table(f"{CATALOG}.{SCHEMA}.products")
# Select business columns only.
product_columns = [
    column for column in products.columns
    if not column.startswith("_")
]

products = products.select(*product_columns)

# Count records before calculating profiling metrics.
product_total_rows = products.count()

if product_total_rows == 0:
    raise ValueError("The Products table is empty. Review ingestion first.")

print(f"Total products: {product_total_rows}")
print(f"Business columns: {len(product_columns)}")

# Inspect stored data types and sample records.
products.printSchema()

display(
    products.orderBy("product_id").limit(10)
)

Total products: 100
Business columns: 4
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_price: string (nullable = true)
 |-- product_category: string (nullable = true)



product_id,product_name,product_price,product_category
P10001,Air purifier,66,electronics
P10002,Air conditioner,74,Electronics
P10003,Alarm clock,71,Electronics
P10004,Backup charger,80,Electronics
P10005,Bread maker,69,Electronics
P10006,Banknote counter,74,Electronics
P10007,Blender,63,Electronics
P10008,Bluetooth speaker,80,Electronics
P10009,Clothes dryer,72,Electronics
P10010,Coffee maker,78,Electronics


### Missing and Distinct Values

- Count nulls and blank text in each business column.
- Calculate missing-value totals and percentages.
- Count distinct populated values exactly as stored.
- Display columns with the highest missing counts first.

Use these findings to prepare completeness rules and understand column uniqueness.

In [0]:
#check missing values and distinct values 
product_missing_results = []
for column in product_columns:

    # Identify actual nulls and blank text.
    is_null = F.col(column).isNull()

    is_blank = (
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) == "")
    )

    null_count = products.filter(is_null).count()
    blank_count = products.filter(is_blank).count()
    missing_count = null_count + blank_count

    # Count distinct populated values exactly as stored.
    distinct_count = (
        products
        .filter(~(is_null | is_blank))
        .select(column)
        .distinct()
        .count()
    )

    product_missing_results.append((
        column,
        null_count,
        blank_count,
        missing_count,
        round(missing_count / product_total_rows * 100, 2),
        distinct_count,
    ))

# Build one summary row per column.
product_column_profile = spark.createDataFrame(
    product_missing_results,
    """
    column_name string,
    null_count long,
    blank_count long,
    missing_count long,
    missing_pct double,
    distinct_values long
    """,
)

print("Results of Missing Check for Products table")

display(
    product_column_profile.orderBy(
        F.desc("missing_count"),
        "column_name",
    )
) # we would expect no misisng vlaues for he whole table especially fpr porduct id 

Results of Missing Check for Products table


column_name,null_count,blank_count,missing_count,missing_pct,distinct_values
product_category,0,0,0,0.0,9
product_id,0,0,0,0.0,100
product_name,0,0,0,0.0,100
product_price,0,0,0,0.0,23


### Duplicate Records and Product IDs

- Count extra copies of identical business records.
- Exclude missing product IDs from repeated-ID checks.
- Count IDs appearing more than once.
- Identify IDs associated with different business records.

These checks help identify ambiguous product records that could affect joins with Sales. No records are removed.

In [0]:
#Check duplicates and repeated product IDs
#counts extra copies of completely identical rows.
product_exact_duplicates  = (
    product_total_rows - products.distinct().count()
)# we expect this value to be 0 as there is no duplicate products with the whole row 

#to check repated products with same id . where the rest of the columns are different. we first rmove null valueso f product_id and remove it's whole row
identified_products = products.filter(
    F.col("product_id").isNotNull()
    &(F.trim(F.col("product_id")) != "")
)

#find ids appear more than one after removeing null vlaues 
repeated_product_ids = (
    identified_products
    .groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
)# there is no products appear more than one

#find IDs associated with different business records 

conflicting_product_ids = (
    identified_products
    .distinct()
    .groupBy("product_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
    .filter(F.col("distinct_versions") > 1)
)


# Summarize duplicate findings.
product_duplicate_profile = spark.createDataFrame(
    [
        ("extra_exact_duplicate_rows", product_exact_duplicates),
        ("repeated_product_id_groups", repeated_product_ids.count()),
        ("conflicting_product_id_groups", conflicting_product_ids.count()),
    ],
    "metric string, value long",
)

display(product_duplicate_profile)

metric,value
extra_exact_duplicate_rows,0
repeated_product_id_groups,0
conflicting_product_id_groups,0


### Inspect Repeated Product Records

- Retrieve the business records belonging to repeated product IDs.
- Display them together to compare their details.
- Use the examples to investigate identical copies or conflicting values.

An empty result means no populated product IDs were repeated.


In [0]:
# Show the business records behind any repeated IDs.
print("Results of Checking duplicates and repeated product IDs")
display(
    identified_products
    .join(
        repeated_product_ids.select("product_id"),
        on="product_id",
        how="inner",
    )
    .select(*product_columns)
    .orderBy("product_id")
)

Results of Checking duplicates and repeated product IDs


product_id,product_name,product_price,product_category


### Price Profiling

Examine whether product prices can be interpreted as numbers and review their distribution. These findings support later price validation and sales-total checks.

#### Prepare and Summarize Prices

- Create a temporary numeric price column while preserving the original values.
- Count converted prices and populated values that failed conversion.
- Count negative and zero prices.
- Calculate the minimum, maximum and average numeric price.

Nulls and failed conversions are excluded from numeric statistics. Negative and zero prices remain included.

In [0]:
#checking product prices 
#We’ll temporarily interpret prices as numbers, then inspect their range and any conversion problems. These results support later price validation and sales-total checks

# Preserve the original price and add a temporary numeric version.
product_price_data = products.withColumn(
    "parsed_price",
    F.expr("try_cast(product_price AS DOUBLE)"),
)

# Identify populated prices that failed conversion.
price_conversion_failed = (
    F.col("product_price").isNotNull()
    & (F.trim(F.col("product_price")) != "")
    & F.col("parsed_price").isNull()
)


# Summarize numeric prices.
product_price_profile = product_price_data.agg(
    F.count("parsed_price").alias("numeric_price_count"),

    F.count(
        F.when(price_conversion_failed, 1)
    ).alias("conversion_failures"),

    F.count(
        F.when(F.col("parsed_price") < 0, 1)
    ).alias("negative_prices"),

    F.count(
        F.when(F.col("parsed_price") == 0, 1)
    ).alias("zero_prices"),

    F.min("parsed_price").alias("minimum_price"),
    F.max("parsed_price").alias("maximum_price"),
    F.avg("parsed_price").alias("average_price"),
)

display(product_price_profile)

numeric_price_count,conversion_failures,negative_prices,zero_prices,minimum_price,maximum_price,average_price
100,0,0,0,10.0,80.0,35.25


#### Inspect Price Issues

- Display populated prices that failed numeric conversion.
- Include numeric prices that are zero or negative.
- Show the product ID, name, original price and converted price.
- Use these examples to investigate potential issues before cleaning.

No prices are corrected or removed.

In [0]:
# Show conversion problems and nonpositive prices for review.
#ther is no covesions problem by the way 
print("Results of Checking product prices ")
display(
    product_price_data
    .filter(
        price_conversion_failed
        | (F.col("parsed_price") <= 0)
    )
    .select(
        "product_id",
        "product_name",
        "product_price",
        "parsed_price",
    )
    .orderBy("product_id")
)

Results of Checking product prices 


product_id,product_name,product_price,parsed_price


### Product Category Distribution

- Count products assigned to each category exactly as stored.
- Calculate each category’s percentage of all products.
- Display the most frequent categories first.
- Review spelling and capitalization differences.

Approved-category checks and alias-based corrections will be handled in later stages.

In [0]:
#check product category 
#category profiling. We list each category exactly as stored and count how many products use it.
# Count products assigned to each raw category value.
product_category_profile = (
    products
    .groupBy("product_category")
    .count()
    .withColumnRenamed("count", "product_count")
    .withColumn(
        "product_pct",
        F.round(
            F.col("product_count") / F.lit(product_total_rows) * 100,
            2,
        ),
    )
)

print("Results of Checking product category")
display(
    product_category_profile.orderBy(
        F.desc("product_count"),
        "product_category",
    )
)

Results of Checking product category


product_category,product_count,product_pct
Clothing,42,42.0
Sports & Outdoors,16,16.0
Pet Supplies,15,15.0
Electronics,12,12.0
Musical Instruments,7,7.0
Home & Kitchen,5,5.0
Clothng,1,1.0
Unknown Category,1,1.0
electronics,1,1.0


### Leading and Trailing Spaces

- Check populated business values for spaces at the beginning or end.
- Exclude nulls and blank-only values already covered by completeness checks.
- Count affected records and display examples when found.
- Show a trimmed preview without changing the original values.

Use these findings to prepare text-standardization rules.

In [0]:
#Check leading and trailing spaces
product_space_results = []
for column in product_columns:

    # Exclude nulls and blank-only values, which were checked earlier.
    has_outer_spaces = (
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) != "")
        & (F.col(column) != F.trim(F.col(column)))
    )

    affected_rows = products.filter(has_outer_spaces).count()

    product_space_results.append(
        (column, affected_rows)
    )

    # Show examples only when a formatting issue exists.
    if affected_rows > 0:
        print(f"Outer spaces found in {column}:")

        display(
            products
            .filter(has_outer_spaces)
            .select(
                F.col("product_id").alias("record_id"),
                F.col(column).alias("original_value"),
                F.trim(F.col(column)).alias("trimmed_preview"),
            )
            .limit(10)
        )

# Display the formatting summary.
product_space_profile = spark.createDataFrame(
    product_space_results,
    "column_name string, rows_with_outer_spaces long",
)

print("Results of Checking product leading and trailing spaces")
display(
    product_space_profile.orderBy(
        F.desc("rows_with_outer_spaces"),
        "column_name",
    )
)


Results of Checking product leading and trailing spaces


column_name,rows_with_outer_spaces
product_category,0
product_id,0
product_name,0
product_price,0


## Stores Profiling

Examine store records for missing values, duplicates, location patterns and text-formatting issues.

Use the findings to prepare store ID and location validation rules. Business columns are checked without modifying the Bronze table.

### Load and Inspect the Data

- Read the Stores table from Bronze.
- Select business columns only.
- Count records and stop if the table is empty.
- Inspect stored data types and display stores ordered by ID.
- Reuse the DataFrame and row count in the following checks.

In [0]:
stores = spark.table(f"{CATALOG}.{SCHEMA}.stores")
# Select business columns only.
store_columns = [
    column for column in stores.columns
    if not column.startswith("_")
]

stores = stores.select(*store_columns)

# Count records before calculating profiling metrics.
store_total_rows = stores.count()

if store_total_rows == 0:
    raise ValueError("The Stores table is empty. Review ingestion first.")

print(f"Total stores: {store_total_rows}")
print(f"Business columns: {len(store_columns)}")

# Inspect stored data types and records.
stores.printSchema()

display(
    stores.orderBy("store_id")
)


Total stores: 14
Business columns: 4
root
 |-- store_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- location: string (nullable = true)



store_id,city,state,location
St_Alb_ N,Albuquerque,New Mexico,Southwest
St_Atl_Ge,Atlanta,Florida,East Coast
St_Bal_Ma,Baltimore,Maryland,East Coast
St_Cha_No,Charlotte,North Carolina,East Coast
St_Col_Oh,Columbus,Ohio,Midwest
St_Det_Mi,Detroit,Michigan,Midwest
St_Hou_Te,Houston,Texas,Southwest
St_Ind_In,Indianapolis,Indiana,Midwest
St_Las_Ne,Las Vegas,Nevada,Far West
St_Los_Ca,Los Angeles,California,Far West


### Missing and Distinct Values

- Count nulls and blank text in each business column.
- Calculate missing-value totals and percentages.
- Count distinct populated values exactly as stored.
- Display columns with the highest missing counts first.

Use these findings to prepare completeness rules and understand column uniqueness.

In [0]:
#Check missing values and distinct values
store_missing_results = []
for column in store_columns:

    # Identify actual nulls and blank text.
    is_null = F.col(column).isNull()

    is_blank = (
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) == "")
    )

    null_count = stores.filter(is_null).count()
    blank_count = stores.filter(is_blank).count()
    missing_count = null_count + blank_count

    # Count distinct populated values exactly as stored.
    distinct_count = (
        stores
        .filter(~(is_null | is_blank))
        .select(column)
        .distinct()
        .count()
    )

    store_missing_results.append((
        column,
        null_count,
        blank_count,
        missing_count,
        round(missing_count / store_total_rows * 100, 2),
        distinct_count,
    ))

# Build one summary row per column.
store_column_profile = spark.createDataFrame(
    store_missing_results,
    """
    column_name string,
    null_count long,
    blank_count long,
    missing_count long,
    missing_pct double,
    distinct_values long
    """,
)

print("Results of Checking store missing values and distinct values")
display(
    store_column_profile.orderBy(
        F.desc("missing_count"),
        "column_name",
    )
)

Results of Checking store missing values and distinct values


column_name,null_count,blank_count,missing_count,missing_pct,distinct_values
city,0,0,0,0.0,14
location,0,0,0,0.0,4
state,0,0,0,0.0,13
store_id,0,0,0,0.0,14


### Duplicate Records and Store IDs

- Count extra copies of identical business records.
- Exclude missing store IDs from repeated-ID checks.
- Count IDs appearing more than once.
- Identify IDs associated with different business records.

These checks help identify ambiguous store records that could affect joins with Sales. No records are removed.

In [0]:
#Check duplicates and repeated store IDs
# Count extra copies of identical business records.
store_exact_duplicates = (
    store_total_rows - stores.distinct().count()
)

# Exclude missing IDs from repeated-ID checks.
identified_stores = stores.filter(
    F.col("store_id").isNotNull()
    & (F.trim(F.col("store_id")) != "")
)

# Find IDs appearing more than once.
repeated_store_ids = (
    identified_stores
    .groupBy("store_id")
    .count()
    .filter(F.col("count") > 1)
)

# Find IDs associated with different business records.
conflicting_store_ids = (
    identified_stores
    .distinct()
    .groupBy("store_id")
    .count()
    .withColumnRenamed("count", "distinct_versions")
    .filter(F.col("distinct_versions") > 1)
)

# Summarize duplicate findings.
store_duplicate_profile = spark.createDataFrame(
    [
        ("extra_exact_duplicate_rows", store_exact_duplicates),
        ("repeated_store_id_groups", repeated_store_ids.count()),
        ("conflicting_store_id_groups", conflicting_store_ids.count()),
    ],
    "metric string, value long",
)

display(store_duplicate_profile)

metric,value
extra_exact_duplicate_rows,0
repeated_store_id_groups,0
conflicting_store_id_groups,0


### Inspect Repeated Store Records

- Retrieve business records belonging to repeated store IDs.
- Display them together to compare their details.
- Use the examples to investigate identical copies or conflicting values.

An empty result means no populated store IDs were repeated.

In [0]:
# Inspect the records behind any repeated IDs.
print("Results of Checking repeated store IDs")
display(
    identified_stores
    .join(
        repeated_store_ids.select("store_id"),
        on="store_id",
        how="inner",
    )
    .select(*store_columns)
    .orderBy("store_id")
)

Results of Checking repeated store IDs


store_id,city,state,location


### City, State and Location Distributions

- Count records for each stored city, state and location value.
- Calculate percentages of all store records.
- Display the most frequent values first.
- Review unexpected labels or inconsistent spelling.

Repeated cities, states or locations are not automatically errors because multiple stores may share them.

In [0]:
#Review city, state and location distributions
# Review each location-related column separately.
for column in ["city", "state", "location"]:

    store_category_profile = (
        stores
        .groupBy(column)
        .count()
        .withColumnRenamed("count", "store_count")
        .withColumn(
            "store_pct",
            F.round(
                F.col("store_count") / F.lit(store_total_rows) * 100,
                2,
            ),
        )
    )

    print(f"Distribution of {column}:")

    display(
        store_category_profile.orderBy(
            F.desc("store_count"),
            column,
        )
    )

Distribution of city:


city,store_count,store_pct
Albuquerque,1,7.14
Atlanta,1,7.14
Baltimore,1,7.14
Charlotte,1,7.14
Columbus,1,7.14
Detroit,1,7.14
Houston,1,7.14
Indianapolis,1,7.14
Las Vegas,1,7.14
Los Angeles,1,7.14


Distribution of state:


state,store_count,store_pct
Florida,2,14.29
Arizona,1,7.14
California,1,7.14
Indiana,1,7.14
Maryland,1,7.14
Michigan,1,7.14
Nevada,1,7.14
New Mexico,1,7.14
North Carolina,1,7.14
Ohio,1,7.14


Distribution of location:


location,store_count,store_pct
Midwest,4,28.57
Southwest,4,28.57
East Coast,3,21.43
Far West,3,21.43


### Inspect Location Combinations

- Display each store’s city, state and location together.
- Review the relationships between these values.
- Prepare for later comparison with the approved store_locations reference.

This view supports inspection; it does not automatically flag incorrect combinations.

In [0]:
#Inspect city–state–location combinations
# Show each store's complete location combination.
store_location_profile = stores.select(
    "store_id",
    "city",
    "state",
    "location",
)
print("Results for city_state-location combinations Test")
display(
    store_location_profile.orderBy("city", "store_id")
)

Results for city_state-location combinations Test


store_id,city,state,location
St_Alb_ N,Albuquerque,New Mexico,Southwest
St_Atl_Ge,Atlanta,Florida,East Coast
St_Bal_Ma,Baltimore,Maryland,East Coast
St_Cha_No,Charlotte,North Carolina,East Coast
St_Col_Oh,Columbus,Ohio,Midwest
St_Det_Mi,Detroit,Michigan,Midwest
St_Hou_Te,Houston,Texas,Southwest
St_Ind_In,Indianapolis,Indiana,Midwest
St_Las_Ne,Las Vegas,Nevada,Far West
St_Los_Ca,Los Angeles,California,Far West


### Leading and Trailing Spaces

- Check populated business values for spaces at the beginning or end.
- Exclude nulls and blank-only values already covered by completeness checks.
- Count affected records and display examples when found.
- Show a trimmed preview without changing the original values.

Internal spaces are preserved. Use the findings to prepare text-standardization rules.

In [0]:
#Check leading and trailing spaces
store_space_results = []

for column in store_columns:

    # Identify outer spaces in populated values.
    has_outer_spaces = (
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) != "")
        & (F.col(column) != F.trim(F.col(column)))
    )

    affected_rows = stores.filter(has_outer_spaces).count()

    store_space_results.append(
        (column, affected_rows)
    )

    # Show examples only when an issue exists.
    if affected_rows > 0:
        print(f"Outer spaces found in {column}:")

        display(
            stores
            .filter(has_outer_spaces)
            .select(
                F.col("store_id").alias("record_id"),
                F.col(column).alias("original_value"),
                F.trim(F.col(column)).alias("trimmed_preview"),
            )
            .limit(10)
        )

# Display the formatting summary.
store_space_profile = spark.createDataFrame(
    store_space_results,
    "column_name string, rows_with_outer_spaces long",
)

print("Results for store space Test")
display(
    store_space_profile.orderBy(
        F.desc("rows_with_outer_spaces"),
        "column_name",
    )
)

Results for store space Test


column_name,rows_with_outer_spaces
city,0
location,0
state,0
store_id,0
